# Decision Tree & Random Forest Fold Results Summary
This notebook summarizes the results of the `*fold_results.csv` files for different window sizes into a single Pandas DataFrame.

In [2]:
import os
import glob
import pandas as pd
import numpy as np
import tkinter as tk
from tkinter import filedialog

### Define Parser Function
We iterate over all files in the `DT` and `RF` directories to extract string definitions and means/standard deviations.

In [ ]:
def process_directory(base_dir):
    # Dynamically find all model folders rather than assuming RF/DT
    possible_models = ['DT', 'RF', 'GB', 'XGB', 'LR', 'SVM']
    models = [m for m in possible_models if os.path.isdir(os.path.join(base_dir, m))]
    
    if not models:
        # Fallback to scanning everything if they used customized names
        dirs = [d for d in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, d))]
        models = [d for d in dirs if d in possible_models]
        
    if not models:
        print("No recognized model directories found in", base_dir)
        return pd.DataFrame()
        
    results = {}
    
    for model in models:
        pattern = os.path.join(base_dir, model, '*', '*fold_results.csv')
        files = glob.glob(pattern)
        
        for file in files:
            filename = os.path.basename(file)
            parts = filename.split('_')
            
            # Extract Dataset Name and Window dynamically
            window = ""
            dataset_name = parts[0] + "_Features"
            
            for i, p in enumerate(parts):
                if p.endswith('s') and p[:-1].isdigit():
                    window = p
                    if i > 0:
                        dataset_name = "_".join(parts[:i])
                    break
            
            df = pd.read_csv(file)
            metrics = ['balanced_accuracy', 'f1_score', 'sensitivity', 'specificity']
            stats = {}
            for m in metrics:
                if m in df.columns:
                    mean_val = df[m].mean()
                    std_val = df[m].std()
                    stats[m] = f"{mean_val:.4f} ± {std_val:.4f}"
                else:
                    stats[m] = "N/A"
                
            features_used = df['features_used'].iloc[0] if 'features_used' in df.columns else "Dynamic Extraction"
            
            key = (dataset_name, window)
            if key not in results:
                results[key] = {'dataset': dataset_name, 'window': window, 'features': features_used}
            
            for m in metrics:
                results[key][f"{m}_{model}"] = stats[m]
                
    # Build dataframe rows dynamically
    rows = []
    for (ds, w), row_data in results.items():
        row = [row_data.get('dataset', ds), row_data.get('window', "")]
        for model in models:
            row.append(row_data.get(f'balanced_accuracy_{model}', ""))
            row.append(row_data.get(f'f1_score_{model}', ""))
            row.append(row_data.get(f'sensitivity_{model}', ""))
            row.append(row_data.get(f'specificity_{model}', ""))
        
        row.append(row_data.get('features', ""))
        row.append("Random undersampling (Auto)") 
        rows.append(row)
        
    # Sort windows
    def window_sort_key(r):
        w = str(r[1]).replace('s','')
        return int(w) if w.isdigit() else 999

        
    rows.sort(key=window_sort_key)
    
    cols = ["dataset", "window"]
    for model in models:
        cols.extend([f"balanced_accuracy_{model}", f"f1_score_{model}", f"sensitivity_{model}", f"specificity_{model}"])
    cols.extend(["feature list", "imbalance handeling"])
    
    out_df = pd.DataFrame(rows, columns=cols)
    return out_df

### Execute and Save
Run the function on the current directory and display the summary.

In [4]:
# Launch Tkinter Directory Picker for Modularity.
root = tk.Tk()
root.withdraw()
root.attributes('-topmost', True)
root.update()

current_dir = filedialog.askdirectory(title="Select extracted_features Directory")

root.update()
root.destroy()

if current_dir:
    print(f"Processing directory: {current_dir}")
    summary_df = process_directory(current_dir)
    
    if not summary_df.empty:
        display(summary_df)

        # Save to CSV using the parent folder explicitly
        csv_path = os.path.join(current_dir, 'model_summary_table.csv')
        summary_df.to_csv(csv_path, index=False)
        print(f"\nSaved dynamic summary to: \n{csv_path}")
    else:
        print("No fold result data found.")
else:
    print("No directory selected.")

Processing directory: /home/wbl-hpc/Desktop/CareWear/carewear-stress-monitoring-analysis-ss-dev-machinelearning/machine_learning/LOPO/Auto_Experiment_Results/1_TimeDomain_Stats
No fold result data found.
